# 07 Topic-Yearly Evolution Heatmap

This notebook reads `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv` and `data/NSFC正式增量采集_37个关键词.csv`, counts annual occurrences and annual prevalence for 37 keyword signals on the fixed `award_year` axis from 2010 to 2023, and exports the Fig. 6 temporal topic heatmap and topic matrix CSV.

Boundary constraint: when run from scratch, this notebook writes only `output/figures/Fig6_temporal_topic_heatmap.*` and `output/tables/07_temporal_topic_matrix.csv`.


## Figure Design Notes

- Core conclusion: BAE-related keyword prevalence changes across `award_year` from 2010 to 2023, separating method, built-environment object, and environmental-performance signals.
- Evidence chain: each heatmap cell reports keyword-matched award records divided by all award records in that year; raw counts are retained in the source matrix CSV.
- Archetype: quantitative grid heatmap, drawn as one complete 37-row panel after sorting keywords by total matched records; each cell is annotated with the raw keyword count, and black outlines mark the three largest non-zero year-over-year absolute prevalence changes for each current year.
- Backend: Python/matplotlib + seaborn only.
- Data sufficiency: each project can match multiple keywords; percentages are keyword-level record prevalence and are not mutually exclusive.
- Export contract: editable SVG/PDF plus high-resolution PNG/TIFF; figure source data is the annual keyword count matrix, with rows ordered by total matched records.


In [ ]:
# Import the core libraries required by this notebook.
# pandas reads and summarizes CSV files; numpy handles matrix operations; matplotlib/seaborn export static charts.
from pathlib import Path
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

# Register common English and CJK fonts; keyword labels come from the formal keyword table and may include Chinese.
FONT_CANDIDATES = [
    "Times New Roman",
    "Arial Unicode MS",
    "PingFang SC",
    "Noto Sans CJK SC",
    "SimHei",
    "DejaVu Sans",
]
FONT_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/PingFang.ttc"),
    Path("/System/Library/Fonts/STHeiti Light.ttc"),
]
for font_path in FONT_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": FONT_CANDIDATES,
    "axes.unicode_minus": False,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Search upward from the current directory for the project root so the notebook works from either the project root or code/ directory.
def find_project_root() -> Path:
    required_files = [
        Path("data") / "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv",
        Path("data") / "NSFC正式增量采集_37个关键词.csv",
    ]
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if all((candidate / rel_path).exists() for rel_path in required_files):
            return candidate.resolve()
    raise FileNotFoundError("Cannot locate the required formal NSFC master data and keyword table from the current working directory.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
KEYWORD_PATH = PROJECT_ROOT / "data" / "NSFC正式增量采集_37个关键词.csv"
FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
TABLE_DIR = PROJECT_ROOT / "output" / "tables"

FIG6_YEAR_START = 2010
FIG6_YEAR_END = 2023
year_index = list(range(FIG6_YEAR_START, FIG6_YEAR_END + 1))

GROUP_ORDER = ["method", "object_built_environment", "performance_environment"]
GROUP_LABELS = {
    "method": "Method",
    "object_built_environment": "Built environment",
    "performance_environment": "Performance",
}

KEYWORD_EN_LABELS = {
    "遥感": "Remote sensing",
    "大数据": "Big data",
    "GIS": "GIS",
    "高分": "High-resolution imagery",
    "多源数据": "Multisource data",
    "三维": "3D",
    "机器学习": "Machine learning",
    "深度学习": "Deep learning",
    "地理信息": "Geoinformation",
    "轨迹": "Trajectory",
    "街景": "Street view",
    "人工智能": "Artificial intelligence",
    "POI": "POI",
    "夜间灯光": "Nighttime lights",
    "手机信令": "Mobile signaling",
    "LiDAR": "LiDAR",
    "LBS": "LBS",
    "规划": "Planning",
    "城市群": "Urban agglomeration",
    "土地利用": "Land use",
    "建筑": "Buildings",
    "绿地": "Green space",
    "街道": "Street",
    "建成环境": "Built environment",
    "街区": "Block",
    "海绵": "Sponge city",
    "城市形态": "Urban morphology",
    "气候": "Climate",
    "生态环境": "Ecological environment",
    "碳": "Carbon",
    "热岛": "Heat island",
    "韧性": "Resilience",
    "热环境": "Thermal environment",
    "暴露": "Exposure",
    "洪涝": "Flooding",
    "能源": "Energy",
    "空气污染": "Air pollution",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Keyword path: {KEYWORD_PATH}")


In [ ]:
# Read the formal incremental master data and 37-keyword table, then run minimal field validation.
# This notebook only reads data and does not write back to or overwrite any raw data.
source_df = pd.read_csv(DATA_PATH)
keywords = pd.read_csv(KEYWORD_PATH)

required_data_columns = {"award_year"}
missing_data_columns = required_data_columns.difference(source_df.columns)
if missing_data_columns:
    raise KeyError(f"Missing required data columns: {sorted(missing_data_columns)}")

required_keyword_columns = {"term", "term_group"}
missing_keyword_columns = required_keyword_columns.difference(keywords.columns)
if missing_keyword_columns:
    raise KeyError(f"Missing required keyword columns: {sorted(missing_keyword_columns)}")

# Convert award_year to integer years; raise immediately for unparseable years to avoid silently dropping records from the matrix.
award_year = pd.to_numeric(source_df["award_year"], errors="coerce")
if award_year.isna().any():
    bad_index = source_df.index[award_year.isna()].tolist()[:10]
    raise ValueError(f"award_year contains unparseable values at rows: {bad_index}")
source_df = source_df.copy()
source_df["award_year"] = award_year.astype(int)

# Fig. 6 uses the fixed complete 2010-2023 award_year axis; if future master data contains out-of-axis years, exclude them from this figure scope and record them in the log.
axis_mask = source_df["award_year"].between(FIG6_YEAR_START, FIG6_YEAR_END)
outside_axis_counts = source_df.loc[~axis_mask, "award_year"].value_counts().sort_index()
df = source_df.loc[axis_mask].copy()
if df.empty:
    raise ValueError("No records remain within the Fig6 award_year axis 2010-2023.")

# Clean the keyword table; order keyword rows first by topic group, then by the table's original order.
keywords = keywords.copy()
keywords["term"] = keywords["term"].astype(str).str.strip()
keywords["term_group"] = keywords["term_group"].astype(str).str.strip()
keywords = keywords[(keywords["term"] != "") & (keywords["term_group"] != "")].copy()
if keywords["term"].duplicated().any():
    duplicated_terms = keywords.loc[keywords["term"].duplicated(), "term"].tolist()
    raise ValueError(f"Duplicated keyword terms: {duplicated_terms}")
if len(keywords) != 37:
    raise ValueError(f"Expected 37 keyword terms, got {len(keywords)}")

keywords["_source_order"] = np.arange(len(keywords))
keywords["_group_order"] = keywords["term_group"].map({group: idx for idx, group in enumerate(GROUP_ORDER)}).fillna(len(GROUP_ORDER)).astype(int)
term_order = keywords.sort_values(["_group_order", "_source_order"]).reset_index(drop=True)
keyword_terms = term_order["term"].tolist()

print(f"Source records: {len(source_df)}")
print(f"Fig6 records within award_year {FIG6_YEAR_START}-{FIG6_YEAR_END}: {len(df)}")
if not outside_axis_counts.empty:
    print("Records outside Fig6 award_year axis:")
    print(outside_axis_counts.to_string())
print(f"Year index: {year_index}")
print(f"Keywords: {len(keyword_terms)}")


In [ ]:
# -----------------------------
# Keyword matching rules
# -----------------------------
# Topic/keyword scope comes entirely from the formal 37-keyword table.
# Each record may match multiple keywords; matrix values represent the number of projects in which that keyword appears in that year.

TEXT_COLS = [
    "project_title", "project_name",
    "abstract_text", "project_abstract_cn", "project_abstract_en", "conclusion_abstract",
    "keywords_raw", "project_keywords",
    "outcomes_text",
    "matched_keywords", "matched_method_terms", "matched_object_terms", "matched_performance_terms",
    "query_matches", "search_payload_keywords", "search_fields",
    "include_reason", "review_flag", "manual_review_flag",
]

# Use only text columns that actually exist in the data so the notebook is robust to minor column differences.
available_text_cols = [col for col in TEXT_COLS if col in df.columns]
if not available_text_cols:
    raise KeyError("No text columns are available for keyword matching.")


def normalize_text(value) -> str:
    """Safely convert a cell to a lowercase string; used only for keyword matching and does not change raw data."""
    if pd.isna(value):
        return ""
    return str(value).lower()


def row_text(row: pd.Series) -> str:
    """Combine multiple text fields from one row, separated by spaces to preserve English word boundaries."""
    parts = [normalize_text(row.get(col, "")) for col in available_text_cols]
    return " ".join(part for part in parts if part)


def compile_keyword_pattern(term: str) -> re.Pattern:
    """Match CJK keywords directly; for terms containing Latin letters or digits, use non-alphanumeric boundaries to reduce substring false positives."""
    escaped = re.escape(term)
    if re.search(r"[A-Za-z0-9]", term):
        return re.compile(rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])", flags=re.IGNORECASE)
    return re.compile(escaped, flags=re.IGNORECASE)


keyword_patterns = {term: compile_keyword_pattern(term) for term in keyword_terms}

df["_keyword_text"] = df.apply(row_text, axis=1)
for term, pattern in keyword_patterns.items():
    df[term] = df["_keyword_text"].str.contains(pattern, regex=True, na=False)

# Quickly inspect overall hits to support manual review of whether keyword rules are too narrow or too broad.
keyword_totals = pd.Series({term: int(df[term].sum()) for term in keyword_terms}, name="matched_awards")
keyword_summary = (
    term_order[["term", "term_group"]]
    .merge(keyword_totals.rename_axis("term").reset_index(), on="term", how="left")
    .sort_values(["term_group", "matched_awards", "term"], ascending=[True, False, True])
)
keyword_summary


In [ ]:
# Generate the year-by-keyword matrix.
# After groupby, reindex to the fixed 2010-2023 annual axis so all years appear in the matrix, figure, and log.
matrix = (
    df.groupby("award_year")[keyword_terms]
    .sum()
    .reindex(year_index, fill_value=0)
    .astype(int)
)
matrix.index.name = "award_year"

year_counts = (
    df.groupby("award_year")
    .size()
    .reindex(year_index, fill_value=0)
    .astype(int)
)
year_counts.name = "award_records"

# Annual prevalence is used for plotting; years with no records are set to NaN to avoid reading no-project years as 0% topic activity.
prevalence_matrix = matrix.div(year_counts.replace(0, np.nan), axis=0) * 100
prevalence_matrix.index.name = "award_year"

# Save the raw count matrix CSV. Column names keep the keyword-table term values so they map directly to figure labels.
TABLE_DIR.mkdir(parents=True, exist_ok=True)
matrix_path = TABLE_DIR / "07_temporal_topic_matrix.csv"
matrix.to_csv(matrix_path, encoding="utf-8-sig")

# Calculate annual prevalence peaks for keywords; peaks are computed only among years with project records.
peak_rows = []
valid_years = year_counts.index[year_counts > 0].tolist()
for term in keyword_terms:
    valid_prevalence = prevalence_matrix.loc[valid_years, term].dropna()
    peak_pct = float(valid_prevalence.max()) if not valid_prevalence.empty else np.nan
    peak_years = valid_prevalence.index[np.isclose(valid_prevalence, peak_pct)].tolist() if not valid_prevalence.empty else []
    peak_n = int(matrix.loc[peak_years, term].max()) if peak_years else 0
    raw_peak_value = int(matrix[term].max())
    raw_peak_years = matrix.index[matrix[term] == raw_peak_value].tolist()
    peak_rows.append({
        "term": term,
        "term_group": term_order.set_index("term").loc[term, "term_group"],
        "peak_pct": peak_pct,
        "peak_pct_years": ", ".join(str(year) for year in peak_years),
        "peak_n_at_peak_pct": peak_n,
        "raw_peak_n": raw_peak_value,
        "raw_peak_years": ", ".join(str(year) for year in raw_peak_years),
        "total_n": int(matrix[term].sum()),
    })
peak_summary = pd.DataFrame(peak_rows)

year_summary = pd.DataFrame({
    "award_year": year_counts.index,
    "award_records": year_counts.values,
    "keyword_matches": matrix.sum(axis=1).values,
})

group_summary = (
    term_order[["term", "term_group"]]
    .merge(keyword_totals.rename_axis("term").reset_index(), on="term", how="left")
    .groupby("term_group", as_index=False)["matched_awards"]
    .sum()
    .sort_values("matched_awards", ascending=False)
)

print(f"Matrix shape: {matrix.shape[0]} years × {matrix.shape[1]} keywords")
display(matrix)
display(year_summary)
display(peak_summary.sort_values("total_n", ascending=False))


In [ ]:
# -----------------------------
# Draw Fig. 6: a single annual keyword heatmap sorted by total hits in descending order
# -----------------------------
# The annual axis uses award_year; keywords are sorted by total 2010-2023 hits in descending order.
# The color scale represents annual keyword prevalence, cell labels show raw counts, and black outlines mark the three non-zero cells with the largest absolute prevalence changes relative to the previous year for each current year.

term_rank = (
    term_order[["term", "term_group", "_source_order"]]
    .merge(keyword_totals.rename_axis("term").reset_index(), on="term", how="left")
    .fillna({"matched_awards": 0})
    .sort_values(["matched_awards", "_source_order"], ascending=[False, True])
    .reset_index(drop=True)
)
ordered_terms = term_rank["term"].tolist()

plot_data = prevalence_matrix[ordered_terms].T.loc[ordered_terms]
plot_counts = matrix[ordered_terms].T.loc[ordered_terms].astype(int)
finite_values = plot_data.to_numpy(dtype=float)
finite_values = finite_values[np.isfinite(finite_values)]
if finite_values.size:
    heatmap_vmax = min(100, max(5, float(np.ceil(np.nanpercentile(finite_values, 95) / 5) * 5)))
else:
    heatmap_vmax = 5

TOKENS = {
    "surface": "#FFFFFF",
    "ink": "#111111",
    "grid": "#F2F2F2",
    "axis": "#111111",
}

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Fig7-derived blue-to-white palette: white background endpoint plus the Fig7 Sankey blue family.
# The blue anchors come from Fig7's visible SVG/code colors for blue Sankey nodes and links.
fig6_heatmap_colors = ["#FFFFFF", "#B5CCDF", "#ABC5DA", "#80A7C8", "#6F9BBF"]
cmap = mpl.colors.LinearSegmentedColormap.from_list("fig7_blue_to_white", fig6_heatmap_colors, N=256)
norm = mpl.colors.Normalize(vmin=0, vmax=heatmap_vmax)

FIG6_FIGSIZE = (9.4, 11.15)
FIG6_AXES_BOUNDS = {"left": 0.300, "right": 0.855, "top": 0.985, "bottom": 0.170}
FIG6_CBAR_BOUNDS = [0.890, 0.18, 0.018, 0.66]
FIG6_FONT_SIZES = {
    "cell_count": 7.1,
    "x_tick": 11.0,
    "y_tick": 10.0,
    "x_label": 14.0,
    "legend": 10.0,
    "cbar_label": 12.0,
    "cbar_tick": 10.6,
}

fig, ax = plt.subplots(figsize=FIG6_FIGSIZE, dpi=180, facecolor=TOKENS["surface"])

def keyword_label(term: str) -> str:
    label = KEYWORD_EN_LABELS.get(term, term)
    total = int(keyword_totals.loc[term])
    return f"{label} ({total:,})"

plot_labels = [keyword_label(term) for term in ordered_terms]
plot_data_labeled = plot_data.copy()
plot_counts_labeled = plot_counts.copy()
plot_data_labeled.index = plot_labels
plot_counts_labeled.index = plot_labels

sns.heatmap(
    plot_data_labeled,
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=heatmap_vmax,
    linewidths=0.32,
    linecolor=TOKENS["grid"],
    cbar=False,
    annot=plot_counts_labeled,
    fmt="d",
    annot_kws={"fontsize": FIG6_FONT_SIZES["cell_count"], "color": "#000000", "fontfamily": "Times New Roman"},
)

term_positions = {term: idx for idx, term in enumerate(ordered_terms)}
prevalence_delta = plot_data.diff(axis=1)
highlight_cells = []
for x_idx, year in enumerate(year_index):
    if x_idx == 0:
        continue
    year_rank = (
        pd.DataFrame({
            "term": ordered_terms,
            "delta": [float(prevalence_delta.loc[term, year]) for term in ordered_terms],
            "term_position": list(range(len(ordered_terms))),
        })
        .assign(abs_delta=lambda d: d["delta"].abs())
        .query("abs_delta > 0 and abs_delta == abs_delta")
        .sort_values(["abs_delta", "term_position"], ascending=[False, True], kind="mergesort")
        .head(3)
    )
    for _, top_row in year_rank.iterrows():
        y_idx = term_positions[top_row["term"]]
        highlight_cells.append({"award_year": year, "term": top_row["term"], "delta": float(top_row["delta"]), "abs_delta": float(top_row["abs_delta"]), "term_position": int(top_row["term_position"])})
        ax.add_patch(Rectangle((x_idx, y_idx), 1, 1, fill=False, edgecolor="#000000", linewidth=0.85, zorder=6))

ax.set_xlabel("Award year", fontsize=FIG6_FONT_SIZES["x_label"], color=TOKENS["axis"], labelpad=13, fontfamily="Times New Roman")
ax.set_ylabel("")
ax.set_xticklabels([str(year) for year in year_index], rotation=90, ha="center", va="top", fontsize=FIG6_FONT_SIZES["x_tick"], fontfamily="Times New Roman")
ax.set_yticklabels(plot_labels, rotation=0, fontsize=FIG6_FONT_SIZES["y_tick"], fontfamily="Times New Roman")
ax.tick_params(axis="x", length=0, colors=TOKENS["axis"], pad=2)
ax.yaxis.tick_left()
ax.tick_params(axis="y", labelleft=True, labelright=False, length=0, colors=TOKENS["axis"], pad=3)

cbar_ax = fig.add_axes(FIG6_CBAR_BOUNDS)
scalar_mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
scalar_mappable.set_array([])
cbar = fig.colorbar(scalar_mappable, cax=cbar_ax, orientation="vertical")
cbar.set_label("Keyword prevalence (%)", fontsize=FIG6_FONT_SIZES["cbar_label"], color=TOKENS["axis"], labelpad=10, fontfamily="Times New Roman")
cbar.ax.tick_params(labelsize=FIG6_FONT_SIZES["cbar_tick"], length=2, colors=TOKENS["axis"])
for tick_label in cbar.ax.get_yticklabels():
    tick_label.set_fontfamily("Times New Roman")

legend_x = 0.105
legend_y = 0.061
legend_box_size = 0.014
fig.add_artist(Rectangle(
    (legend_x, legend_y + 0.014),
    legend_box_size,
    legend_box_size,
    transform=fig.transFigure,
    fill=False,
    edgecolor="#000000",
    linewidth=0.85,
    clip_on=False,
    zorder=20,
))
fig.text(
    legend_x + 0.022,
    legend_y + 0.021,
    "Black outlines mark the three keywords with the largest absolute year-over-year change in annual prevalence for each year (2011-2023).",
    ha="left",
    va="center",
    fontsize=FIG6_FONT_SIZES["legend"],
    color=TOKENS["ink"],
    fontfamily="Times New Roman",
)
fig.text(
    legend_x + 0.022,
    legend_y + 0.003,
    "No outlines are shown for 2010 because no previous year is available.",
    ha="left",
    va="center",
    fontsize=FIG6_FONT_SIZES["legend"],
    color=TOKENS["ink"],
    fontfamily="Times New Roman",
)

fig.subplots_adjust(**FIG6_AXES_BOUNDS)

# Export four formats. SVG/PDF provide editable vectors, and PNG/TIFF support preview and high-resolution document reuse.
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
figure_base = FIGURE_DIR / "Fig6_temporal_topic_heatmap"
export_paths = {
    "svg": figure_base.with_suffix(".svg"),
    "pdf": figure_base.with_suffix(".pdf"),
    "png": figure_base.with_suffix(".png"),
    "tiff": figure_base.with_suffix(".tiff"),
}

fig.savefig(export_paths["svg"], bbox_inches="tight")
fig.savefig(export_paths["pdf"], bbox_inches="tight")
fig.savefig(export_paths["png"], dpi=600, bbox_inches="tight")
fig.savefig(export_paths["tiff"], dpi=600, bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
plt.show()

export_paths


In [ ]:
# Generate the results paragraph.
# The paragraph is displayed only in the notebook and is not written to a log or draft file.
top_keywords = keyword_totals.sort_values(ascending=False).head(3)
top_groups = group_summary.head(3)
max_year = int(year_counts.idxmax())
max_year_n = int(year_counts.loc[max_year])

paragraph = (
    f"Fig. 6 is based on {len(df)} project records in the formal incremental master table with award_year from {FIG6_YEAR_START} to {FIG6_YEAR_END}. "
    f"It maps 37 keywords as a keyword-by-year heatmap, and the x-axis fully covers {FIG6_YEAR_START}-{FIG6_YEAR_END}. "
    "The color scale represents each keyword's share of award records in that year. "
    f"The three most frequently matched keywords are {top_keywords.index[0]} (N={int(top_keywords.iloc[0])}), "
    f"{top_keywords.index[1]} (N={int(top_keywords.iloc[1])}) and {top_keywords.index[2]} (N={int(top_keywords.iloc[2])}). "
    f"By keyword group, {GROUP_LABELS.get(top_groups.iloc[0]['term_group'], top_groups.iloc[0]['term_group'])} is the strongest signal (N={int(top_groups.iloc[0]['matched_awards'])}). "
    f"Annual project volume peaks in {max_year} (n={max_year_n}); 2022 and 2023 contain {int(year_counts.loc[2022])} and {int(year_counts.loc[2023])} records, respectively, and both are included on the annual axis according to the observed award_year."
)

print(paragraph)


In [ ]:
# Final output check.
# Check each required file for this task to ensure it exists and is non-empty after a full notebook run.
expected_files = [
    FIGURE_DIR / "Fig6_temporal_topic_heatmap.svg",
    FIGURE_DIR / "Fig6_temporal_topic_heatmap.pdf",
    FIGURE_DIR / "Fig6_temporal_topic_heatmap.tiff",
    FIGURE_DIR / "Fig6_temporal_topic_heatmap.png",
    TABLE_DIR / "07_temporal_topic_matrix.csv",
]

file_audit = []
for path in expected_files:
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    file_audit.append({"file": str(path.relative_to(PROJECT_ROOT)), "exists": exists, "bytes": size})
    assert exists and size > 0, f"Expected output missing or empty: {path}"

pd.DataFrame(file_audit)
